In [16]:
import pandas as pd
import re
from collections import Counter

In [17]:
df = pd.read_csv('../Sentiment_Analysis.csv', encoding='latin1')
df = df.drop(['textID', 'Time of Tweet', 'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)'], axis=1)
df.rename(columns={"sentiment": "label"}, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4815 entries, 0 to 4814
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    3534 non-null   object
 1   label   3534 non-null   object
dtypes: object(2)
memory usage: 75.4+ KB


df = pd.read_csv('twitter_training.csv', header=None, names=['id', 'topic', 'label', 'text'])
df =df.drop(['id', 'topic'], axis=1)
df['label'] = df['label'].str.strip().str.lower()

In [18]:
def preprocess(text):
    if pd.isna(text):
        return []
    text = re.sub(r'[^a-z\s]', '', str(text).lower())
    return text.split()

In [19]:
pos_words = Counter()
neg_words = Counter()

In [20]:
for _, row in df.iterrows():
    words = preprocess(row["text"])
    label = str(row["label"]).strip().lower()
    if label == "positive":
        pos_words.update(words)
    elif label == "negative":
        neg_words.update(words)

In [21]:
print("✅ Positive word count:", len(pos_words))
print("❌ Negative word count:", len(neg_words))

print("\nTop 20 positive words:")
print(pos_words.most_common(20))

print("\nTop 20 negative words:")
print(neg_words.most_common(20))

✅ Positive word count: 3226
❌ Negative word count: 3095

Top 20 positive words:
[('i', 512), ('the', 359), ('to', 354), ('a', 273), ('you', 265), ('and', 201), ('my', 182), ('it', 176), ('for', 162), ('good', 143), ('day', 139), ('love', 135), ('is', 134), ('in', 131), ('so', 117), ('of', 112), ('that', 100), ('happy', 98), ('be', 98), ('was', 97)]

Top 20 negative words:
[('i', 589), ('to', 340), ('the', 315), ('my', 272), ('a', 230), ('and', 205), ('im', 169), ('it', 159), ('is', 157), ('me', 131), ('you', 131), ('in', 130), ('of', 119), ('so', 116), ('for', 114), ('have', 112), ('on', 111), ('that', 105), ('not', 91), ('its', 90)]


In [22]:
def classify_sentence(text, pos_words, neg_words):
    words = preprocess(text)
    pos_score = 0
    neg_score = 0

    for w in words:
        if w in pos_words:
            pos_score += pos_words[w]
        if w in neg_words:
            neg_score += neg_words[w]

    if pos_score > neg_score:
        return "Positive 😊"
    elif neg_score > pos_score:
        return "Negative 😠"
    else:
        return "Neutral 😐"


In [23]:
examples = [
    "This game is so good that i want to play it again some more",
    "I hate this so much, it’s terrible and broken.",
    "It’s okay, not bad but not amazing either."
]

for text in examples:
    result = classify_sentence(text, pos_words, neg_words)
    print(f"{text} → {result}")

This game is so good that i want to play it again some more → Positive 😊
I hate this so much, it’s terrible and broken. → Negative 😠
It’s okay, not bad but not amazing either. → Negative 😠
